# Lesson 35: Graphical Models & Hidden Markov Models

*ML Academy — companion notebook*

Bayesian networks and explaining away, hidden Markov models, the forward and Viterbi algorithms, learning parameters, and stationary distributions.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-35.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Bayesian networks

In [1]:
import itertools
import numpy as np
# A small Bayesian network: Rain → Sprinkler, and both → WetGrass.
P_rain = {1: 0.2, 0: 0.8}
P_sprinkler = {1: {1: 0.01, 0: 0.99}, 0: {1: 0.4, 0: 0.6}}          # P(sprinkler | rain)
P_wet = {(1, 1): 0.99, (1, 0): 0.9, (0, 1): 0.8, (0, 0): 0.0}         # P(wet = 1 | sprinkler, rain)
def joint(r, s, w):
    pw = P_wet[(s, r)]
    return P_rain[r] * P_sprinkler[r][s] * (pw if w else 1 - pw)
def query(target, evidence):
    num = den = 0.0
    for r, s, w in itertools.product([0, 1], repeat=3):
        vals = {"rain": r, "sprinkler": s, "wet": w}
        if all(vals[k] == v for k, v in evidence.items()):
            p = joint(r, s, w); den += p
            if vals[target] == 1: num += p
    return num / den
print("P(rain)                         ", round(P_rain[1], 3))
print("P(rain | grass is wet)          ", round(query("rain", {"wet": 1}), 3))
print("P(rain | wet, sprinkler was on) ", round(query("rain", {"wet": 1, "sprinkler": 1}), 3))

P(rain)                          0.2
P(rain | grass is wet)           0.358
P(rain | wet, sprinkler was on)  0.007


## 3. Hidden Markov models

In [2]:
# Two-state weather HMM: hidden Sunny/Rainy, observed activity Walk/Shop/Clean.
states, obs_names = ["Sunny", "Rainy"], ["Walk", "Shop", "Clean"]
pi = np.array([0.6, 0.4])
T = np.array([[0.8, 0.2], [0.3, 0.7]])                               # T[i, j] = P(next j | now i)
E = np.array([[0.6, 0.3, 0.1], [0.1, 0.4, 0.5]])                     # E[i, k] = P(observe k | state i)
def forward(obs):
    alpha = pi * E[:, obs[0]]
    for o in obs[1:]:
        alpha = (alpha @ T) * E[:, o]
    return alpha.sum()
week = [0, 0, 1, 2, 2, 1, 0]                                         # Walk Walk Shop Clean Clean Shop Walk
print("P(this week's activities) under the model:", f"{forward(week):.6f}")
print("same activities, shuffled order:", f"{forward([2, 0, 2, 0, 1, 0, 1]):.6f}")

P(this week's activities) under the model: 0.000555
same activities, shuffled order: 0.000255


## 4. Decoding: the Viterbi algorithm

In [3]:
def viterbi(obs):
    logd = np.log(pi) + np.log(E[:, obs[0]]); back = []
    for o in obs[1:]:
        cand = logd[:, None] + np.log(T)                             # cand[i, j]: best path ending i, then j
        back.append(cand.argmax(axis=0)); logd = cand.max(axis=0) + np.log(E[:, o])
    path = [int(logd.argmax())]
    for b in reversed(back):
        path.append(int(b[path[-1]]))
    return path[::-1]
print("activities:", [obs_names[o] for o in week])
print("most likely weather:", [states[s] for s in viterbi(week)])

activities: ['Walk', 'Walk', 'Shop', 'Clean', 'Clean', 'Shop', 'Walk']
most likely weather: ['Sunny', 'Sunny', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Sunny']


## 6. Learning an HMM's parameters

In [4]:
rng = np.random.default_rng(0)
def sample(n):
    z = [rng.choice(2, p=pi)]
    for _ in range(n - 1): z.append(rng.choice(2, p=T[z[-1]]))
    return np.array(z), np.array([rng.choice(3, p=E[s]) for s in z])
z_train, o_train = sample(5000)
# Learn the parameters by counting, from labelled sequences
T_hat = np.array([[np.mean(z_train[1:][z_train[:-1] == i] == j) for j in range(2)] for i in range(2)])
E_hat = np.array([[np.mean(o_train[z_train == i] == k) for k in range(3)] for i in range(2)])
print("learned transitions:", T_hat.round(2).tolist(), "  true:", T.tolist())
print("learned emissions:  ", E_hat.round(2).tolist())

learned transitions: [[0.79, 0.21], [0.31, 0.69]]   true: [[0.8, 0.2], [0.3, 0.7]]
learned emissions:   [[0.59, 0.31, 0.1], [0.11, 0.39, 0.5]]


## 7. Why the sequence matters

In [5]:
z_test, o_test = sample(2000)
best_guess = np.argmax(E, axis=0)[o_test]                           # most likely state for each day alone
acc_alone = (best_guess == z_test).mean()
acc_vit = (np.array(viterbi(list(o_test))) == z_test).mean()
print("each day on its own:", round(acc_alone, 3), "  Viterbi using the sequence:", round(acc_vit, 3))
stationary = np.linalg.matrix_power(T, 100)[0]
print("long-run share of sunny days:", round(stationary[0], 3), "  observed in the sample:", round((z_test == 0).mean(), 3))

each day on its own: 0.703   Viterbi using the sequence: 0.752
long-run share of sunny days: 0.6   observed in the sample: 0.595


## 9. In Python: tracking a machine's hidden health

In [6]:
import numpy as np

pi = np.array([0.95, 0.05])                    # healthy, worn
T = np.array([[0.97, 0.03], [0.0, 1.0]])       # wear never repairs itself
E = np.array([[0.9, 0.1], [0.3, 0.7]])         # P(normal, alarm | state)
readings = [0, 0, 1, 0, 0, 1, 1, 0, 1, 1]      # 0 normal, 1 alarm
belief = pi * E[:, readings[0]]; belief /= belief.sum()
history = [belief[1]]
for r in readings[1:]:
    belief = (belief @ T) * E[:, r]; belief /= belief.sum()
    history.append(belief[1])
print("readings:        ", readings)
print("P(worn) after each:", [round(float(p), 2) for p in history])
print("first reading where P(worn) > 0.5:", int(np.argmax(np.array(history) > 0.5)) + 1)

readings:         [0, 0, 1, 0, 0, 1, 1, 0, 1, 1]
P(worn) after each: [0.02, 0.02, 0.25, 0.11, 0.05, 0.38, 0.82, 0.61, 0.92, 0.99]
first reading where P(worn) > 0.5: 7


## 12. Going deeper: forward–backward and Baum–Welch

In [7]:
import numpy as np
pi = np.array([0.6, 0.4]); T = np.array([[0.8, 0.2], [0.3, 0.7]]); E = np.array([[0.6, 0.3, 0.1], [0.1, 0.4, 0.5]])
obs = [0, 1, 1, 2, 1, 0]                                   # Walk Shop Shop Clean Shop Walk
def forward_backward(o):
    n = len(o); a = np.zeros((n, 2)); b = np.ones((n, 2))
    a[0] = pi * E[:, o[0]]
    for t in range(1, n): a[t] = (a[t - 1] @ T) * E[:, o[t]]
    for t in range(n - 2, -1, -1): b[t] = T @ (E[:, o[t + 1]] * b[t + 1])
    post = a * b
    return post / post.sum(axis=1, keepdims=True)
print("P(rainy) each day:", np.round(forward_backward(obs)[:, 1], 3).tolist())

P(rainy) each day: [0.14, 0.417, 0.588, 0.741, 0.488, 0.157]


In [8]:
rng = np.random.default_rng(0)
z = [0]
for _ in range(2999): z.append(rng.choice(2, p=T[z[-1]]))
o = np.array([rng.choice(3, p=E[s]) for s in z])
# Baum–Welch from the observations only, starting from a rough guess
Th, Eh, ph = np.array([[0.6, 0.4], [0.4, 0.6]]), np.array([[0.4, 0.3, 0.3], [0.2, 0.3, 0.5]]), np.array([0.5, 0.5])
for it in range(50):
    n = len(o); a = np.zeros((n, 2)); b = np.ones((n, 2)); c = np.zeros(n)
    a[0] = ph * Eh[:, o[0]]; c[0] = a[0].sum(); a[0] /= c[0]
    for t in range(1, n): a[t] = (a[t - 1] @ Th) * Eh[:, o[t]]; c[t] = a[t].sum(); a[t] /= c[t]
    for t in range(n - 2, -1, -1): b[t] = Th @ (Eh[:, o[t + 1]] * b[t + 1]) / c[t + 1]
    gam = a * b
    xi = np.einsum("ti,ij,tj->ij", a[:-1], Th, Eh[:, o[1:]].T * b[1:] / c[1:, None])
    Th = xi / xi.sum(axis=1, keepdims=True)
    Eh = np.array([[gam[o == k, i].sum() for k in range(3)] for i in range(2)]) / gam.sum(axis=0)[:, None]
    ph = gam[0]
print("log-likelihood after 50 iterations:", round(float(np.log(c).sum()), 1))
print("learned transitions:", np.round(Th, 2).tolist())
print("learned emissions:  ", np.round(Eh, 2).tolist())

log-likelihood after 50 iterations: -3208.4
learned transitions: [[0.75, 0.25], [0.25, 0.75]]
learned emissions:   [[0.65, 0.29, 0.07], [0.16, 0.39, 0.45]]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
df["metro"] = df.description.str.contains("metro").astype(int)
# A tiny Bayesian network learned by counting: Area → Metro, Area → SoldFast
p_area = df.area.value_counts(normalize=True)
p_metro = df.groupby("area").metro.mean(); p_fast = df.groupby("area").sold_fast.mean()
num = sum(p_area[a] * p_metro[a] * p_fast[a] for a in p_area.index)
den = sum(p_area[a] * p_metro[a] for a in p_area.index)
print("P(sold fast):", round(df.sold_fast.mean(), 3), "  P(sold fast | near metro), via area:", round(num / den, 3))
print("P(area | near metro):", {a: round(float(p_area[a] * p_metro[a] / den), 2) for a in p_area.index})

P(sold fast): 0.5   P(sold fast | near metro), via area: 0.75
P(area | near metro): {'Suburb': 0.0, 'Central': 1.0, 'Outskirts': 0.0}


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Bayes in a two-node network

Compute P(cause | effect).

In [11]:
p_cause = 0.02
p_eff_given_cause, p_eff_given_not = 0.8, 0.1
post = p_eff_given_cause  # ← apply Bayes' theorem
check(post, 0.14035087719298245, hint='Prior × likelihood, divided by the total probability of the effect.')

❌ Not yet. Hint: Prior × likelihood, divided by the total probability of the effect.


### Challenge 2: One step of a Markov chain

Compute tomorrow's state probabilities.

In [12]:
import numpy as np
today = np.array([0.5, 0.5])
T = np.array([[0.9, 0.1], [0.2, 0.8]])
tomorrow = T @ today  # ← the row vector goes on the left
result = np.round(tomorrow, 2).tolist()
check(result, [0.55, 0.45], hint='Multiply the row vector by the transition matrix.')

❌ Not yet. Hint: Multiply the row vector by the transition matrix.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Bayes in a two-node network** — the missing line:

```python
post = p_cause * p_eff_given_cause / (p_cause * p_eff_given_cause + (1 - p_cause) * p_eff_given_not)
```

**2. One step of a Markov chain** — the missing line:

```python
tomorrow = today @ T
```